Current Version: Taxable Cost Basis Upgrade (in progress)
Next step: verify taxable withdrawal gain calculation

# Retirement Simulator Starter Notebook

This notebook is a starter skeleton for a year-by-year retirement simulator.
It is designed to model:
- you + spouse
- earned income
- rental income
- pension income
- Social Security income
- annual spending
- taxes (simplified federal model)
- RMD
- Roth conversions
- account balances over time

The notebook is intentionally modular so you can improve one piece at a time.

## 1. Imports

In [1]:
from dataclasses import dataclass
from typing import Optional, List, Dict, Any
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.0f}'.format)

## 2. Input Schema and Data Classes

These classes define the conceptual input model.
You can start by editing the sample data in the next section.

In [2]:
@dataclass
class Person:
    name: str
    birth_year: int


@dataclass
class IncomeStream:
    name: str
    owner: str  # 'you', 'spouse', or 'joint'
    income_type: str  # salary, rental, pension, social_security, other
    start_year: int
    end_year: int
    annual_amount: float
    cola_rate: float = 0.0
    taxable_fraction: float = 1.0  # simplified; SS can be improved later


@dataclass
class Account:
    name: str
    account_type: str  # taxable, traditional, roth, cash
    owner: str  # 'you', 'spouse', or 'joint'
    start_balance: float
    annual_return: float
    cost_basis: float = 0.0   # only meaningful for taxable accounts


@dataclass
class SpendingPlan:
    base_spending: float
    inflation_rate: float
    special_spending_by_year: Optional[Dict[int, float]] = None


@dataclass
class TaxSettings:
    filing_status: str
    standard_deduction: float
    ordinary_brackets: List[Dict[str, float]]
    capital_gains_brackets: List[Dict[str, float]]
    state_tax_rate: float = 0.0


@dataclass
class RMDSettings:
    start_age: int
    divisors_by_age: Dict[int, float]


@dataclass
class RothStrategy:
    strategy_type: str  # none, fixed, fill_bracket, custom
    fixed_amount: float = 0.0
    bracket_top: Optional[float] = None
    custom_by_year: Optional[Dict[int, float]] = None

    # NEW FIELDS
    start_year: Optional[int] = None
    end_year: Optional[int] = None
    stop_at_rmd: bool = False


@dataclass
class WithdrawalStrategy:
    order: List[str]  # e.g. ['cash', 'taxable', 'traditional', 'roth']


@dataclass
class SimulationSettings:
    current_year: int
    start_year: int
    end_year: int


@dataclass
class Scenario:
    you: Person
    spouse: Person
    simulation: SimulationSettings
    income_streams: List[IncomeStream]
    accounts: List[Account]
    spending: SpendingPlan
    taxes: TaxSettings
    rmd: RMDSettings
    roth_strategy: RothStrategy
    withdrawal_strategy: WithdrawalStrategy

## 3. Sample Inputs

Replace these placeholder values with your own assumptions.

In [3]:
you = Person(name='You', birth_year=1965)
spouse = Person(name='Spouse', birth_year=1963)

simulation = SimulationSettings(
    current_year=2026,
    start_year=2026,
    end_year=2060,
)

# Simplified example tax brackets for MFJ; replace with your preferred values.
ordinary_brackets = [
    {'top': 23850, 'rate': 0.10},
    {'top': 96950, 'rate': 0.12},
    {'top': 206700, 'rate': 0.22},
    {'top': 394600, 'rate': 0.24},
    {'top': 501050, 'rate': 0.32},
    {'top': 751600, 'rate': 0.35},
    {'top': float('inf'), 'rate': 0.37},
]
capital_gains_brackets = [
    {'top': 100000, 'rate': 0.00},      # placeholder: replace with your chosen 0% cap gains top
    {'top': 600000, 'rate': 0.15},      # placeholder: replace with your chosen 15% cap gains top
    {'top': float('inf'), 'rate': 0.20},
]
taxes = TaxSettings(
    filing_status='married_filing_jointly',
    standard_deduction=30000,
    ordinary_brackets=ordinary_brackets,
    capital_gains_brackets=capital_gains_brackets,
    state_tax_rate=0.045,
)

rmd_divisors = {
    73: 26.5, 74: 25.5, 75: 24.6, 76: 23.7, 77: 22.9,
    78: 22.0, 79: 21.1, 80: 20.2, 81: 19.4, 82: 18.5,
    83: 17.7, 84: 16.8, 85: 16.0, 86: 15.2, 87: 14.4,
    88: 13.7, 89: 12.9, 90: 12.2,
}

rmd = RMDSettings(start_age=73, divisors_by_age=rmd_divisors)

income_streams = [
    IncomeStream(name='Spouse Salary', owner='spouse', income_type='salary', start_year=2026, end_year=2027, annual_amount=200000, cola_rate=0.03),
    IncomeStream(name='Rental Income', owner='joint',income_type='rental', start_year=2026, end_year=2060,annual_amount=120000,cola_rate=0.02,taxable_fraction=0.1),
    IncomeStream(name='Your Pension', owner='you', income_type='pension', start_year=2029, end_year=2060, annual_amount=24000, cola_rate=0.00),
    IncomeStream(name='Your Social Security', owner='you', income_type='social_security', start_year=2031, end_year=2060, annual_amount=40870, cola_rate=0.02, taxable_fraction=0.85),
    IncomeStream(name='Spouse Social Security', owner='spouse', income_type='social_security', start_year=2028, end_year=2060, annual_amount=39600, cola_rate=0.02, taxable_fraction=0.85),
]

accounts = [
    Account(
    name='Taxable Brokerage',
    account_type='taxable',
    owner='joint',
    start_balance=1300000,
    annual_return=0.05,
    cost_basis=600000
),
    Account(name='Traditional IRA', account_type='traditional', owner='you', start_balance=4600000, annual_return=0.05),
    Account(name='Roth IRA', account_type='roth', owner='you', start_balance=700000, annual_return=0.05),
    Account(name='Cash', account_type='cash', owner='joint', start_balance=450000, annual_return=0.02),
]

@dataclass
class CarReplacementPlan:
    first_replacement_year: int
    replacement_interval_years: int
    purchase_amount: float
    inflation_rate: float = 0.0
    last_replacement_year: Optional[int] = None


def build_car_replacement_schedule(plan: CarReplacementPlan, simulation: SimulationSettings) -> Dict[int, float]:
    schedule = {}

    end_year = (
        plan.last_replacement_year
        if plan.last_replacement_year is not None
        else simulation.end_year
    )

    year = plan.first_replacement_year
    replacement_count = 0

    while year <= min(end_year, simulation.end_year):
        amount = plan.purchase_amount * ((1 + plan.inflation_rate) ** replacement_count)
        schedule[year] = amount
        year += plan.replacement_interval_years
        replacement_count += 1

    return schedule


def merge_special_spending_events(*event_dicts: Dict[int, float]) -> Dict[int, float]:
    merged = {}

    for event_dict in event_dicts:
        for year, amount in event_dict.items():
            merged[year] = merged.get(year, 0.0) + amount

    return merged


mortgage_schedule = {
    2026: 55200,
    2027: 55200,
    2028: 55200,
    2029: 55200,
    2030: 55200,
    2031: 55200,
    2032: 55200,
    2033: 55200,
    2034: 55200,
    2035: 55200,
    2036: 27600
}

car_plan = CarReplacementPlan(
    first_replacement_year=2030,
    replacement_interval_years=8,
    purchase_amount=45000,
    inflation_rate=0.025
)

car_schedule = build_car_replacement_schedule(car_plan, simulation)

special_spending = merge_special_spending_events(
    mortgage_schedule,
    car_schedule
)

spending = SpendingPlan(
    base_spending=140000,
    inflation_rate=0.025,
    special_spending_by_year=special_spending,
)

roth_strategy = RothStrategy(
    strategy_type='fill_bracket',
    bracket_top=394600,
    start_year=2026,
    end_year=2037,
    stop_at_rmd=True
)
withdrawal_strategy = WithdrawalStrategy(
    order=['cash', 'taxable', 'traditional', 'roth']
)

scenario = Scenario(
    you=you,
    spouse=spouse,
    simulation=simulation,
    income_streams=income_streams,
    accounts=accounts,
    spending=spending,
    taxes=taxes,
    rmd=rmd,
    roth_strategy=roth_strategy,
    withdrawal_strategy=withdrawal_strategy,
)

## 4. Helper Functions

In [4]:
def age_in_year(birth_year: int, year: int) -> int:
    return year - birth_year


def inflate(amount: float, rate: float, years_from_start: int) -> float:
    return amount * ((1 + rate) ** years_from_start)


def get_account_map(accounts: List[Account]) -> Dict[str, Account]:
    return {a.account_type: a for a in accounts}

In [5]:
def calculate_taxable_withdrawal_components(withdrawal: float, balance: float, cost_basis: float) -> Dict[str, float]:
    """
    Split a taxable brokerage withdrawal into:
    - principal returned
    - realized capital gain
    - updated account balance
    - updated cost basis
    """
    if withdrawal <= 0 or balance <= 0:
        return {
            'principal_portion': 0.0,
            'capital_gain_portion': 0.0,
            'new_balance': balance,
            'new_cost_basis': cost_basis,
        }

    withdrawal = min(withdrawal, balance)

    unrealized_gain = max(0.0, balance - cost_basis)
    gain_ratio = unrealized_gain / balance if balance > 0 else 0.0

    capital_gain_portion = withdrawal * gain_ratio
    principal_portion = withdrawal - capital_gain_portion

    new_balance = balance - withdrawal
    new_cost_basis = max(0.0, cost_basis - principal_portion)

    return {
        'principal_portion': principal_portion,
        'capital_gain_portion': capital_gain_portion,
        'new_balance': new_balance,
        'new_cost_basis': new_cost_basis,
    }

In [6]:
def calculate_taxable_social_security(
    filing_status: str,
    social_security_income: float,
    other_income_for_ss_formula: float
) -> Dict[str, float]:
    """
    Approximate taxable Social Security using IRS-style combined income thresholds.

    combined_income =
        other_income_for_ss_formula
        + 0.5 * social_security_income

    For MFJ:
        base1 = 32000
        base2 = 44000

    For single:
        base1 = 25000
        base2 = 34000
    """
    if social_security_income <= 0:
        return {
            'combined_income': other_income_for_ss_formula,
            'taxable_social_security': 0.0,
            'taxable_social_security_fraction': 0.0,
        }

    if filing_status == 'married_filing_jointly':
        base1 = 32000.0
        base2 = 44000.0
    else:
        base1 = 25000.0
        base2 = 34000.0

    combined_income = other_income_for_ss_formula + 0.5 * social_security_income

    if combined_income <= base1:
        taxable_ss = 0.0
    elif combined_income <= base2:
        taxable_ss = min(
            0.5 * social_security_income,
            0.5 * (combined_income - base1)
        )
    else:
        taxable_ss = min(
            0.85 * social_security_income,
            0.85 * (combined_income - base2) + min(6000.0, 0.5 * social_security_income)
            if filing_status == 'married_filing_jointly'
            else 0.85 * (combined_income - base2) + min(4500.0, 0.5 * social_security_income)
        )

    taxable_fraction = taxable_ss / social_security_income if social_security_income > 0 else 0.0

    return {
        'combined_income': combined_income,
        'taxable_social_security': taxable_ss,
        'taxable_social_security_fraction': taxable_fraction,
    }

In [7]:
def generate_bracket_targets(min_income, max_income, step):
    return list(range(min_income, max_income + step, step))

In [8]:
def compute_after_tax_total(df, tax_rate=0.24):
    end = df.iloc[-1]

    after_tax_total = (
        end["end_taxable"]
        + end["end_roth"]
        + end["end_traditional"] * (1 - tax_rate)
    )

    return after_tax_total

## 5. Income Engine

In [9]:
def compute_income_for_year(year: int, income_streams: List[IncomeStream]) -> Dict[str, float]:
    result = {
        'salary_you': 0.0,
        'salary_spouse': 0.0,
        'rental_income': 0.0,
        'pension_you': 0.0,
        'pension_spouse': 0.0,
        'ss_you': 0.0,
        'ss_spouse': 0.0,
        'other_income': 0.0,
        'gross_nonportfolio_income': 0.0,
        'taxable_nonportfolio_income': 0.0,
    }

    for stream in income_streams:
        if stream.start_year <= year <= stream.end_year:
            years_from_start = year - stream.start_year
            amount = inflate(stream.annual_amount, stream.cola_rate, years_from_start)

            if stream.income_type == 'salary' and stream.owner == 'you':
                result['salary_you'] += amount
                result['taxable_nonportfolio_income'] += amount

            elif stream.income_type == 'salary' and stream.owner == 'spouse':
                result['salary_spouse'] += amount
                result['taxable_nonportfolio_income'] += amount

            elif stream.income_type == 'rental':
                result['rental_income'] += amount
                result['taxable_nonportfolio_income'] += amount * stream.taxable_fraction

            elif stream.income_type == 'pension' and stream.owner == 'you':
                result['pension_you'] += amount
                result['taxable_nonportfolio_income'] += amount

            elif stream.income_type == 'pension' and stream.owner == 'spouse':
                result['pension_spouse'] += amount
                result['taxable_nonportfolio_income'] += amount

            elif stream.income_type == 'social_security' and stream.owner == 'you':
                result['ss_you'] += amount
                # DO NOT add to taxable_nonportfolio_income here

            elif stream.income_type == 'social_security' and stream.owner == 'spouse':
                result['ss_spouse'] += amount
                # DO NOT add to taxable_nonportfolio_income here

            else:
                result['other_income'] += amount
                result['taxable_nonportfolio_income'] += amount * stream.taxable_fraction

            result['gross_nonportfolio_income'] += amount

    return result

## 6. Spending Engine

In [10]:
def compute_spending_for_year(year: int, start_year: int, spending_plan: SpendingPlan) -> Dict[str, float]:
    years_from_start = year - start_year
    base_spending = inflate(spending_plan.base_spending, spending_plan.inflation_rate, years_from_start)
    special = 0.0
    if spending_plan.special_spending_by_year:
        special = spending_plan.special_spending_by_year.get(year, 0.0)
    return {
        'base_spending': base_spending,
        'special_spending': special,
        'total_spending': base_spending + special,
    }

## 7. RMD Engine

In [11]:
def compute_rmd(age: int, prior_year_end_traditional: float, rmd_settings: RMDSettings) -> float:
    if age < rmd_settings.start_age:
        return 0.0
    divisor = rmd_settings.divisors_by_age.get(age)
    if divisor is None or divisor <= 0:
        return 0.0
    return prior_year_end_traditional / divisor

## 8. Roth Conversion Strategy

In [12]:
def choose_roth_conversion(
    year: int,
    rmd_required: float,
    strategy: RothStrategy,
    current_ordinary_income_before_conversion: float,
    available_traditional_balance: float,
) -> float:
    """
    Dynamic Roth conversion decision engine.

    current_ordinary_income_before_conversion should be taxable ORDINARY income
    before Roth conversion is added.
    """

    if strategy.start_year is not None and year < strategy.start_year:
        return 0.0

    if strategy.end_year is not None and year > strategy.end_year:
        return 0.0

    if strategy.stop_at_rmd and rmd_required > 0:
        return 0.0

    if strategy.strategy_type == 'none':
        return 0.0

    if strategy.strategy_type == 'fixed':
        return min(strategy.fixed_amount, available_traditional_balance)

    if strategy.strategy_type == 'custom':
        amount = 0.0 if not strategy.custom_by_year else strategy.custom_by_year.get(year, 0.0)
        return min(amount, available_traditional_balance)

    if strategy.strategy_type == 'fill_bracket':
        if strategy.bracket_top is None:
            return 0.0

        room = max(0.0, strategy.bracket_top - current_ordinary_income_before_conversion)
        return min(room, available_traditional_balance)

    return 0.0

## 9. Simplified Tax Engine

This is a basic ordinary income tax calculator.
You can replace it later with a more detailed tax model.

In [13]:
def compute_federal_tax(taxable_income: float, brackets: List[Dict[str, float]]) -> float:
    if taxable_income <= 0:
        return 0.0

    tax = 0.0
    lower = 0.0
    remaining = taxable_income

    for bracket in brackets:
        top = bracket['top']
        rate = bracket['rate']
        width = top - lower
        taxed_here = min(remaining, width)
        if taxed_here > 0:
            tax += taxed_here * rate
            remaining -= taxed_here
        lower = top
        if remaining <= 0:
            break

    return tax

def compute_capital_gains_tax(
    taxable_ordinary_income: float,
    taxable_capital_gains: float,
    capital_gains_brackets: List[Dict[str, float]]
) -> float:
    """
    Tax long-term capital gains using stacked brackets.
    Ordinary income fills the lower brackets first.
    """
    if taxable_capital_gains <= 0:
        return 0.0

    tax = 0.0
    remaining_gains = taxable_capital_gains
    lower = 0.0
    occupied_income = taxable_ordinary_income

    for bracket in capital_gains_brackets:
        top = bracket['top']
        rate = bracket['rate']

        bracket_start = max(lower, occupied_income)
        room = max(0.0, top - bracket_start)

        taxed_here = min(remaining_gains, room)
        if taxed_here > 0:
            tax += taxed_here * rate
            remaining_gains -= taxed_here

        lower = top

        if remaining_gains <= 0:
            break

    return tax


def compute_taxes(
    gross_ordinary_income: float,
    gross_capital_gains_income: float,
    tax_settings: TaxSettings
) -> Dict[str, float]:
    """
    Split taxes into:
    - ordinary income tax
    - long-term capital gains tax

    Standard deduction is applied to ordinary income first, then any remaining deduction
    reduces capital gains.
    """
    deduction = tax_settings.standard_deduction

    taxable_ordinary_income = max(0.0, gross_ordinary_income - deduction)
    remaining_deduction = max(0.0, deduction - gross_ordinary_income)

    taxable_capital_gains = max(0.0, gross_capital_gains_income - remaining_deduction)

    federal_ordinary_tax = compute_federal_tax(
        taxable_ordinary_income,
        tax_settings.ordinary_brackets
    )

    federal_capital_gains_tax = compute_capital_gains_tax(
        taxable_ordinary_income=taxable_ordinary_income,
        taxable_capital_gains=taxable_capital_gains,
        capital_gains_brackets=tax_settings.capital_gains_brackets
    )

    federal_tax = federal_ordinary_tax + federal_capital_gains_tax

    # Simplified state tax: apply to total taxable income
    taxable_income_total = taxable_ordinary_income + taxable_capital_gains
    state_tax = taxable_income_total * tax_settings.state_tax_rate

    total_tax = federal_tax + state_tax

    return {
        'gross_ordinary_income': gross_ordinary_income,
        'gross_capital_gains_income': gross_capital_gains_income,
        'gross_taxable_income': gross_ordinary_income + gross_capital_gains_income,

        'deduction': deduction,
        'taxable_ordinary_income': taxable_ordinary_income,
        'taxable_capital_gains': taxable_capital_gains,
        'taxable_income': taxable_income_total,

        'federal_ordinary_tax': federal_ordinary_tax,
        'federal_capital_gains_tax': federal_capital_gains_tax,
        'federal_tax': federal_tax,
        'state_tax': state_tax,
        'total_tax': total_tax,
    }

## 10. Withdrawal Engine

This function funds spending and taxes using your selected withdrawal order.
It treats Roth conversion separately from cash withdrawals.

In [14]:
def fund_cash_need(cash_need: float, balances: Dict[str, float], withdrawal_order: List[str]) -> Dict[str, float]:
    remaining_need = max(0.0, cash_need)
    withdrawals = {
        'withdrawal_cash': 0.0,
        'withdrawal_taxable': 0.0,
        'withdrawal_traditional_cash': 0.0,
        'withdrawal_roth': 0.0,
    }

    mapping = {
        'cash': 'withdrawal_cash',
        'taxable': 'withdrawal_taxable',
        'traditional': 'withdrawal_traditional_cash',
        'roth': 'withdrawal_roth',
    }

    for source in withdrawal_order:
        if remaining_need <= 0:
            break

        available = balances.get(source, 0.0)
        take = min(available, remaining_need)
        withdrawals[mapping[source]] += take
        balances[source] = available - take
        remaining_need -= take

    withdrawals['unfunded_need'] = remaining_need
    return withdrawals

## 11. One-Year Simulation Function

This is the key transition function for one year.

In [15]:
def run_one_year(
    year: int,
    scenario: Scenario,
    start_balances: Dict[str, float],
    prior_year_end_traditional: float,
    start_taxable_cost_basis: float,
) -> Dict[str, Any]:
    you_age = age_in_year(scenario.you.birth_year, year)
    spouse_age = age_in_year(scenario.spouse.birth_year, year)

    income = compute_income_for_year(year, scenario.income_streams)
    spending = compute_spending_for_year(year, scenario.simulation.start_year, scenario.spending)

    total_social_security_income = income['ss_you'] + income['ss_spouse']

    # Income that counts toward the SS combined-income formula, excluding SS itself
    other_income_for_ss_formula = income['taxable_nonportfolio_income']

    ss_tax_info = calculate_taxable_social_security(
        filing_status=scenario.taxes.filing_status,
        social_security_income=total_social_security_income,
        other_income_for_ss_formula=other_income_for_ss_formula
    )

    taxable_social_security = ss_tax_info['taxable_social_security']
    combined_income = ss_tax_info['combined_income']
    taxable_social_security_fraction = ss_tax_info['taxable_social_security_fraction']

    rmd_required = compute_rmd(you_age, prior_year_end_traditional, scenario.rmd)

    projected_ordinary_income_before_conversion = max(
        0.0,
        income['taxable_nonportfolio_income']
        + taxable_social_security
    )
    # Amount of traditional balance eligible for Roth conversion
    # (RMD portion cannot be converted)
    convertible_traditional_balance = max(
        0.0,
        start_balances['traditional'] - rmd_required
    )
    roth_conversion = choose_roth_conversion(
        year=year,
        rmd_required=rmd_required,
        strategy=scenario.roth_strategy,
        current_ordinary_income_before_conversion=projected_ordinary_income_before_conversion,
        available_traditional_balance=convertible_traditional_balance,
    )
  # First-pass taxes, before realizing taxable-account gains from withdrawals
    gross_ordinary_income = (
        income['taxable_nonportfolio_income']
        + taxable_social_security
        + roth_conversion
    )

    gross_capital_gains_income = 0.0

    taxes = compute_taxes(
      gross_ordinary_income=gross_ordinary_income,
      gross_capital_gains_income=gross_capital_gains_income,
      tax_settings=scenario.taxes
    )

    cash_need = max(
        0.0,
        spending['total_spending'] + taxes['total_tax'] - income['gross_nonportfolio_income']
    )

    balances_for_withdrawal = {
        'cash': start_balances['cash'],
        'taxable': start_balances['taxable'],
        'traditional': max(0.0, start_balances['traditional'] - roth_conversion),
        'roth': start_balances['roth'],
    }

    withdrawals = fund_cash_need(
        cash_need,
        balances_for_withdrawal,
        scenario.withdrawal_strategy.order
    )

    # --- NEW: taxable brokerage gain realization ---
    taxable_withdrawal_info = calculate_taxable_withdrawal_components(
        withdrawal=withdrawals['withdrawal_taxable'],
        balance=start_balances['taxable'],
        cost_basis=start_taxable_cost_basis,
    )

    realized_capital_gain = taxable_withdrawal_info['capital_gain_portion']
    taxable_principal_returned = taxable_withdrawal_info['principal_portion']
    taxable_cost_basis_after_withdrawal = taxable_withdrawal_info['new_cost_basis']

    # Update taxable balance to match helper result
    balances_for_withdrawal['taxable'] = taxable_withdrawal_info['new_balance']

    pretax_distribution_total = roth_conversion + withdrawals['withdrawal_traditional_cash']
    forced_distribution = 0.0

    rmd_shortfall = max(0.0, rmd_required - pretax_distribution_total)

    if rmd_shortfall > 0:
        forced_distribution = min(balances_for_withdrawal['traditional'], rmd_shortfall)
        withdrawals['withdrawal_traditional_cash'] += forced_distribution
        balances_for_withdrawal['traditional'] -= forced_distribution
        balances_for_withdrawal['cash'] += forced_distribution           # ADD THIS LINE
        pretax_distribution_total += forced_distribution
        rmd_shortfall = max(0.0, rmd_required - pretax_distribution_total)

    # Recompute taxes including:
    # - Roth conversion
    # - traditional cash withdrawals
    # - realized capital gains from taxable sales
    # Separate ordinary income and capital gains (future upgrade)
    gross_ordinary_income = (
        income['taxable_nonportfolio_income']
        + taxable_social_security
        + roth_conversion
        + withdrawals['withdrawal_traditional_cash']
    )

    # TODO: capital gains should be taxed separately
    gross_capital_gains_income = realized_capital_gain
    gross_taxable_income = gross_ordinary_income + gross_capital_gains_income

    taxes = compute_taxes(
      gross_ordinary_income=gross_ordinary_income,
      gross_capital_gains_income=gross_capital_gains_income,
      tax_settings=scenario.taxes
    )

    # Recompute cash need because taxes may have changed
    cash_need = max(
        0.0,
        spending['total_spending'] + taxes['total_tax'] - income['gross_nonportfolio_income']
    )

    current_cash_withdrawals = (
        withdrawals['withdrawal_cash']
        + withdrawals['withdrawal_taxable']
        + withdrawals['withdrawal_traditional_cash']
        + withdrawals['withdrawal_roth']
    )

    additional_cash_need = max(0.0, cash_need - current_cash_withdrawals)

    if additional_cash_need > 0:
        extra_withdrawals = fund_cash_need(
            additional_cash_need,
            balances_for_withdrawal,
            scenario.withdrawal_strategy.order
        )

        # If extra taxable withdrawals occur, compute extra gains too
        if extra_withdrawals['withdrawal_taxable'] > 0:
            extra_taxable_info = calculate_taxable_withdrawal_components(
                withdrawal=extra_withdrawals['withdrawal_taxable'],
                balance=balances_for_withdrawal['taxable'],
                cost_basis=taxable_cost_basis_after_withdrawal,
            )

            realized_capital_gain += extra_taxable_info['capital_gain_portion']
            taxable_principal_returned += extra_taxable_info['principal_portion']
            taxable_cost_basis_after_withdrawal = extra_taxable_info['new_cost_basis']
            balances_for_withdrawal['taxable'] = extra_taxable_info['new_balance']

        withdrawals['withdrawal_cash'] += extra_withdrawals['withdrawal_cash']
        withdrawals['withdrawal_taxable'] += extra_withdrawals['withdrawal_taxable']
        withdrawals['withdrawal_traditional_cash'] += extra_withdrawals['withdrawal_traditional_cash']
        withdrawals['withdrawal_roth'] += extra_withdrawals['withdrawal_roth']
        withdrawals['unfunded_need'] = extra_withdrawals['unfunded_need']

        pretax_distribution_total = roth_conversion + withdrawals['withdrawal_traditional_cash']
        rmd_shortfall = max(0.0, rmd_required - pretax_distribution_total)

        gross_ordinary_income = (
          income['taxable_nonportfolio_income']
          + taxable_social_security
          + roth_conversion
          + withdrawals['withdrawal_traditional_cash']
        )

        gross_capital_gains_income = realized_capital_gain

        taxes = compute_taxes(
            gross_ordinary_income=gross_ordinary_income,
            gross_capital_gains_income=gross_capital_gains_income,
            tax_settings=scenario.taxes
        )
    else:
        withdrawals['unfunded_need'] = withdrawals.get('unfunded_need', 0.0)

    total_cash_inflows = (
        income['gross_nonportfolio_income']
        + withdrawals['withdrawal_cash']
        + withdrawals['withdrawal_taxable']
        + withdrawals['withdrawal_traditional_cash']
        + withdrawals['withdrawal_roth']
    )

    total_cash_outflows = spending['total_spending'] + taxes['total_tax']
    net_cash_before_reinvestment = total_cash_inflows - total_cash_outflows

    pre_growth_taxable = balances_for_withdrawal['taxable']
    pre_growth_traditional = balances_for_withdrawal['traditional']
    pre_growth_roth = start_balances['roth'] + roth_conversion - withdrawals['withdrawal_roth']
    pre_growth_cash = balances_for_withdrawal['cash'] + net_cash_before_reinvestment

    account_map = get_account_map(scenario.accounts)
    return_taxable = account_map['taxable'].annual_return
    return_traditional = account_map['traditional'].annual_return
    return_roth = account_map['roth'].annual_return
    return_cash = account_map['cash'].annual_return

    growth_taxable = pre_growth_taxable * return_taxable
    growth_traditional = pre_growth_traditional * return_traditional
    growth_roth = pre_growth_roth * return_roth
    growth_cash = pre_growth_cash * return_cash

    end_taxable = pre_growth_taxable + growth_taxable
    end_traditional = pre_growth_traditional + growth_traditional
    end_roth = pre_growth_roth + growth_roth
    end_cash = pre_growth_cash + growth_cash
    end_total = end_taxable + end_traditional + end_roth + end_cash

    # Simplified assumption:
    # growth in taxable account increases unrealized gains, not cost basis
    end_taxable_cost_basis = taxable_cost_basis_after_withdrawal

    return {
        'year': year,
        'you_age': you_age,
        'spouse_age': spouse_age,
        'start_taxable': start_balances['taxable'],
        'start_traditional': start_balances['traditional'],
        'start_roth': start_balances['roth'],
        'start_cash': start_balances['cash'],
        'start_total': sum(start_balances.values()),
        'start_taxable_cost_basis': start_taxable_cost_basis,
        **income,
        **spending,
        'combined_income': combined_income,
        'taxable_social_security': taxable_social_security,
        'taxable_social_security_fraction': taxable_social_security_fraction,
        'rmd_required': rmd_required,
        'roth_conversion': roth_conversion,
        'realized_capital_gain': realized_capital_gain,
        'taxable_principal_returned': taxable_principal_returned,
        **taxes,
        **withdrawals,
        'pretax_distribution_total': pretax_distribution_total,
        'rmd_shortfall': rmd_shortfall,
        'total_cash_inflows': total_cash_inflows,
        'total_cash_outflows': total_cash_outflows,
        'net_cash_before_reinvestment': net_cash_before_reinvestment,
        'pre_growth_taxable': pre_growth_taxable,
        'pre_growth_traditional': pre_growth_traditional,
        'pre_growth_roth': pre_growth_roth,
        'pre_growth_cash': pre_growth_cash,
        'return_taxable': return_taxable,
        'return_traditional': return_traditional,
        'return_roth': return_roth,
        'return_cash': return_cash,
        'growth_taxable': growth_taxable,
        'growth_traditional': growth_traditional,
        'growth_roth': growth_roth,
        'growth_cash': growth_cash,
        'end_taxable': end_taxable,
        'end_traditional': end_traditional,
        'end_roth': end_roth,
        'end_cash': end_cash,
        'end_total': end_total,
        'end_taxable_cost_basis': end_taxable_cost_basis,
        # NEW LINE
        'taxable_account_unrealized_gain': end_taxable - end_taxable_cost_basis,
        **taxes,
    }

## 12. Full Simulation Loop

In [16]:
def run_simulation(scenario: Scenario) -> pd.DataFrame:
    account_map = get_account_map(scenario.accounts)

    balances = {
        'taxable': account_map['taxable'].start_balance,
        'traditional': account_map['traditional'].start_balance,
        'roth': account_map['roth'].start_balance,
        'cash': account_map['cash'].start_balance,
    }

    taxable_cost_basis = account_map['taxable'].cost_basis

    rows = []
    prior_year_end_traditional = balances['traditional']

    for year in range(scenario.simulation.start_year, scenario.simulation.end_year + 1):
        row = run_one_year(
            year=year,
            scenario=scenario,
            start_balances=balances.copy(),
            prior_year_end_traditional=prior_year_end_traditional,
            start_taxable_cost_basis=taxable_cost_basis,
        )
        rows.append(row)

        balances = {
            'taxable': row['end_taxable'],
            'traditional': row['end_traditional'],
            'roth': row['end_roth'],
            'cash': row['end_cash'],
        }

        taxable_cost_basis = row['end_taxable_cost_basis']
        prior_year_end_traditional = row['end_traditional']

    return pd.DataFrame(rows)

## 13. Run Baseline Scenario

In [17]:
df = run_simulation(scenario)
df.head(10)

,year,you_age,spouse_age,start_taxable,start_traditional,start_roth,start_cash,start_total,start_taxable_cost_basis,salary_you,salary_spouse,rental_income,pension_you,pension_spouse,ss_you,ss_spouse,other_income,gross_nonportfolio_income,taxable_nonportfolio_income,base_spending,special_spending,total_spending,combined_income,taxable_social_security,taxable_social_security_fraction,rmd_required,roth_conversion,realized_capital_gain,taxable_principal_returned,gross_ordinary_income,gross_capital_gains_income,gross_taxable_income,deduction,taxable_ordinary_income,taxable_capital_gains,taxable_income,federal_ordinary_tax,federal_capital_gains_tax,federal_tax,state_tax,total_tax,withdrawal_cash,withdrawal_taxable,withdrawal_traditional_cash,withdrawal_roth,unfunded_need,pretax_distribution_total,rmd_shortfall,total_cash_inflows,total_cash_outflows,net_cash_before_reinvestment,pre_growth_taxable,pre_growth_traditional,pre_growth_roth,pre_growth_cash,return_taxable,return_traditional,return_roth,return_cash,growth_taxable,growth_traditional,growth_roth,growth_cash,end_taxable,end_traditional,end_roth,end_cash,end_total,end_taxable_cost_basis,taxable_account_unrealized_gain
0,2026,61,63,"1,300,000","4,600,000","700,000","450,000","7,050,000","600,000",0,"200,000","120,000",0,0,0,0,0,"320,000","212,000","140,000","55,200","195,200","212,000",0,0,0,"182,600",0,0,"394,600",0,"394,600",30000,"364,600",0,"364,600","73,198",0,"73,198","16,407","89,605",0,0,0,0,0,"182,600",0,"320,000","284,805","35,195","1,300,000","4,417,400","882,600","485,195",0,0,0,0,"65,000","220,870","44,130","9,704","1,365,000","4,638,270","926,730","494,899","7,424,899","600,000","765,000"
1,2027,62,64,"1,365,000","4,638,270","926,730","494,899","7,424,899","600,000",0,"206,000","122,400",0,0,0,0,0,"328,400","218,240","143,500","55,200","198,700","218,240",0,0,0,"176,360",0,0,"394,600",0,"394,600",30000,"364,600",0,"364,600","73,198",0,"73,198","16,407","89,605",0,0,0,0,0,"176,360",0,"328,400","288,305","40,095","1,365,000","4,461,910","1,103,090","534,994",0,0,0,0,"68,250","223,096","55,154","10,700","1,433,250","4,685,006","1,158,244","545,694","7,822,194","600,000","833,250"
2,2028,63,65,"1,433,250","4,685,006","1,158,244","545,694","7,822,194","600,000",0,0,"124,848",0,0,0,"39,600",0,"164,448","12,485","147,088","55,200","202,288","32,285",142,0,0,"381,973",0,0,"394,600",0,"394,600",30000,"364,600",0,"364,600","73,198",0,"73,198","16,407","89,605","127,444",0,0,0,0,"381,973",0,"291,892","291,892",0,"1,433,250","4,303,033","1,540,217","418,249",0,0,0,0,"71,662","215,152","77,011","8,365","1,504,912","4,518,184","1,617,228","426,614","8,066,939","600,000","904,912"
3,2029,64,66,"1,504,912","4,518,184","1,617,228","426,614","8,066,939","600,000",0,0,"127,345","24,000",0,0,"40,392",0,"191,737","36,734","150,765","55,200","205,965","56,930","16,991",0,0,"340,875",0,0,"394,600",0,"394,600",30000,"364,600",0,"364,600","73,198",0,"73,198","16,407","89,605","103,833",0,0,0,0,"340,875",0,"295,570","295,570",0,"1,504,912","4,177,310","1,958,103","322,782",0,0,0,0,"75,246","208,865","97,905","6,456","1,580,158","4,386,175","2,056,008","329,237","8,351,578","600,000","980,158"
4,2030,65,67,"1,580,158","4,386,175","2,056,008","329,237","8,351,578","600,000",0,0,"129,892","24,000",0,0,"41,200",0,"195,092","36,989","154,534","100,200","254,734","57,589","17,551",0,0,"340,060",0,0,"394,600",0,"394,600",30000,"364,600",0,"364,600","73,198",0,"73,198","16,407","89,605","149,247",0,0,0,0,"340,060",0,"344,339","344,339",0,"1,580,158","4,046,115","2,396,068","179,990",0,0,0,0,"79,008","202,306","119,803","3,600","1,659,166","4,248,421","2,515,871","183,590","8,607,048","600,000","1,059,166"
5,2031,66,68,"1,659,166","4,248,421","2,515,871","183,590","8,607,048","600,000",0,0,"132,490","24,000",0,"40,870","42,024",0,"239,384","37,249","158,397","55,200","213,597","78,696","35,492",0,0,"321,860",0,0,"394,600",0,"394,600",30000,"364,600",0,"364,600","73,198",0,"73,198","16,407","89,605","

In [18]:


pd.DataFrame(
    [{'year': y, 'special_spending': a} for y, a in sorted(special_spending.items())]
)

,year,special_spending
0,2026,"55,200"
1,2027,"55,200"
2,2028,"55,200"
3,2029,"55,200"
4,2030,"100,200"
5,2031,"55,200"
6,2032,"55,200"
7,2033,"55,200"
8,2034,"55,200"
9,2035,"55,200"


In [19]:
def evaluate_roth_targets(
    base_scenario: Scenario,
    targets: List[float],
    strategy_type: str = 'fill_bracket',
    start_year: Optional[int] = None,
    end_year: Optional[int] = None,
    stop_at_rmd: bool = True
) -> pd.DataFrame:
    results = []

    for target in targets:
        test_strategy = RothStrategy(
          strategy_type=strategy_type,
          bracket_top=target,
          start_year=start_year,
          end_year=end_year,
          stop_at_rmd=stop_at_rmd
        )

        test_scenario = clone_scenario_with_roth_strategy(base_scenario, test_strategy)
        df_test = run_simulation(test_scenario)

        results.append({
            'bracket_top': target,
            'final_total_assets': df_test['end_total'].iloc[-1],
            'final_taxable': df_test['end_taxable'].iloc[-1],
            'final_traditional': df_test['end_traditional'].iloc[-1],
            'final_roth': df_test['end_roth'].iloc[-1],
            'final_cash': df_test['end_cash'].iloc[-1],
            'total_federal_tax': df_test['federal_tax'].sum(),
            'total_state_tax': df_test['state_tax'].sum(),
            'total_tax': df_test['total_tax'].sum(),
            'total_roth_conversion': df_test['roth_conversion'].sum(),
            'first_rmd_year': df_test.loc[df_test['rmd_required'] > 0, 'year'].min()
                if (df_test['rmd_required'] > 0).any() else None,
            'max_rmd': df_test['rmd_required'].max(),
            'ending_traditional_balance': df_test['end_traditional'].iloc[-1],
            'ending_roth_balance': df_test['end_roth'].iloc[-1],
             # ✅ NEW (this is what you optimize)
            'final_after_tax_assets': compute_after_tax_total(df_test),
        })

    return pd.DataFrame(results)

In [20]:
def evaluate_roth_conversion_sweep(base_scenario, conversion_amounts):
    results = []

    for amount in conversion_amounts:

        test_strategy = RothStrategy(
            strategy_type="fixed",
            fixed_amount=amount
        )

        # Save original strategy
        original_strategy = base_scenario.roth_strategy

        # Apply new strategy
        base_scenario.roth_strategy = test_strategy

        df_test = run_simulation(base_scenario)

        results.append({
            'conversion_amount': amount,

            'final_total_assets': df_test['end_total'].iloc[-1],
            'final_after_tax_assets': compute_after_tax_total(df_test),

            'total_tax': df_test['total_tax'].sum(),
            'max_rmd': df_test['rmd_required'].max(),
        })

        # print(df_test[['end_traditional', 'end_roth', 'end_taxable', 'end_cash', 'end_total']].tail())

        # Restore original strategy (VERY IMPORTANT)
        base_scenario.roth_strategy = original_strategy

    return pd.DataFrame(results)

In [25]:
def evaluate_spending_levels(base_scenario, spending_levels):
    results = []

    for spend in spending_levels:

        df_test = run_simulation(base_scenario, annual_spending=spend)

        results.append({
            'spending': spend,
            'final_total_assets': df_test['end_total'].iloc[-1],
            'final_after_tax_assets': compute_after_tax_total(df_test),
            'total_tax': df_test['total_tax'].sum(),
            'max_rmd': df_test['rmd_required'].max(),
        })

    return pd.DataFrame(results)

In [26]:
conversion_amounts = list(range(0, 450001, 50000))
sweep_view = evaluate_roth_conversion_sweep(
    base_scenario=scenario,
    conversion_amounts=conversion_amounts
)

In [27]:
spending_levels = [120000, 140000, 160000, 180000]

spending_view = evaluate_spending_levels(
    base_scenario=scenario,
    spending_levels=spending_levels
)

display(spending_view)

TypeError: run_simulation() got an unexpected keyword argument 'annual_spending'

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(sweep_view['conversion_amount'], sweep_view['final_total_assets'], marker='o', label="Pre-Tax")
plt.plot(sweep_view['conversion_amount'], sweep_view['final_after_tax_assets'], marker='o', label="After-Tax")

plt.legend()
plt.title('Pre-Tax vs After-Tax Assets by Roth Conversion Amount')
plt.xlabel('Annual Roth Conversion Amount')
plt.ylabel('Assets')
plt.grid(True)

plt.show()

In [ ]:
# Scenario comparison dashboard

def clone_scenario_with_roth_strategy(base_scenario, roth_strategy):
    return Scenario(
        you=base_scenario.you,
        spouse=base_scenario.spouse,
        simulation=base_scenario.simulation,
        income_streams=base_scenario.income_streams,
        accounts=base_scenario.accounts,
        spending=base_scenario.spending,
        taxes=base_scenario.taxes,
        rmd=base_scenario.rmd,
        roth_strategy=roth_strategy,
        withdrawal_strategy=base_scenario.withdrawal_strategy,
    )

# Define scenarios
scenario_none = clone_scenario_with_roth_strategy(
    scenario,
    RothStrategy(strategy_type='none')
)

scenario_fixed_100k = clone_scenario_with_roth_strategy(
    scenario,
    RothStrategy(strategy_type='fixed', fixed_amount=100000)
)

scenario_fill_24 = clone_scenario_with_roth_strategy(
    scenario,
    RothStrategy(strategy_type='fill_bracket', bracket_top=394600)
)

# Run simulations
df_none = run_simulation(scenario_none)
df_fixed_100k = run_simulation(scenario_fixed_100k)
df_fill_24 = run_simulation(scenario_fill_24)

# Build summary metrics
def build_scenario_summary(df, scenario_name):
    return {
        'scenario': scenario_name,
        'final_total_assets': df['end_total'].iloc[-1],
        'final_taxable': df['end_taxable'].iloc[-1],
        'final_traditional': df['end_traditional'].iloc[-1],
        'final_roth': df['end_roth'].iloc[-1],
        'final_cash': df['end_cash'].iloc[-1],
        'total_federal_tax': df['federal_tax'].sum(),
        'total_state_tax': df['state_tax'].sum(),
        'total_tax': df['total_tax'].sum(),
        'total_roth_conversion': df['roth_conversion'].sum(),
        'first_rmd_year': df.loc[df['rmd_required'] > 0, 'year'].min() if (df['rmd_required'] > 0).any() else None,
        'max_rmd': df['rmd_required'].max(),
        'ending_traditional_balance': df['end_traditional'].iloc[-1],
        'ending_roth_balance': df['end_roth'].iloc[-1],
    }

comparison_summary = pd.DataFrame([
    build_scenario_summary(df_none, 'No Conversion'),
    build_scenario_summary(df_fixed_100k, 'Fixed 100k/yr'),
    build_scenario_summary(df_fill_24, 'Fill 24% Bracket'),
])

comparison_summary

In [ ]:
def optimize_roth_bracket(
    base_scenario: Scenario,
    min_income: int = 50000,
    max_income: int = 500000,
    step: int = 10000,
    start_year: Optional[int] = None,
    end_year: Optional[int] = None,
    stop_at_rmd: bool = True
):

    targets = generate_bracket_targets(min_income, max_income, step)

    results = evaluate_roth_targets(
        base_scenario=base_scenario,
        targets=targets,
        start_year=start_year,
        end_year=end_year,
        stop_at_rmd=stop_at_rmd
    )

    best_tax = results.sort_values('total_tax').iloc[0]
    best_assets = results.sort_values('final_total_assets', ascending=False).iloc[0]

    return {
        "all_results": results,
        "best_tax_strategy": best_tax,
        "best_asset_strategy": best_assets
    }

In [ ]:
optimizer_view = evaluate_roth_targets(
    base_scenario=scenario,
    targets=[96950, 206700, 394600, 501050],
    start_year=2026,
    end_year=2037,
    stop_at_rmd=True
)

optimizer_view

In [ ]:
optimizer = optimize_roth_bracket(
    base_scenario=scenario,
    min_income=50000,
    max_income=450000,
    step=10000,
    start_year=2026,
    end_year=2037,
    stop_at_rmd=True
)

In [ ]:
optimizer["best_asset_strategy"]

In [ ]:
results = optimizer["all_results"]

plt.figure(figsize=(10,5))
plt.plot(results['bracket_top'], results['total_tax'])
plt.title("Lifetime Tax vs Roth Conversion Target")
plt.xlabel("Bracket Target")
plt.ylabel("Total Tax")
plt.grid(True)
plt.show()

In [ ]:
optimizer_view.sort_values('total_tax')

In [ ]:
optimizer_view.sort_values('final_total_assets', ascending=False)

In [ ]:
optimizer_view.sort_values('max_rmd')

In [ ]:
def evaluate_fixed_roth_amounts(
    base_scenario: Scenario,
    amounts: List[float],
    start_year: Optional[int] = None,
    end_year: Optional[int] = None,
    stop_at_rmd: bool = False
) -> pd.DataFrame:
    results = []

    for amount in amounts:
        test_strategy = RothStrategy(
            strategy_type='fixed',
            fixed_amount=amount,
            start_year=start_year,
            end_year=end_year,
            stop_at_rmd=stop_at_rmd
        )

        test_scenario = clone_scenario_with_roth_strategy(base_scenario, test_strategy)
        df_test = run_simulation(test_scenario)

        results.append({
            'fixed_amount': amount,
            'final_total_assets': df_test['end_total'].iloc[-1],
            'total_tax': df_test['total_tax'].sum(),
            'max_rmd': df_test['rmd_required'].max(),
            'ending_traditional_balance': df_test['end_traditional'].iloc[-1],
            'ending_roth_balance': df_test['end_roth'].iloc[-1],
            'total_roth_conversion': df_test['roth_conversion'].sum(),
        })

    return pd.DataFrame(results)

In [ ]:
fixed_optimizer_view = evaluate_fixed_roth_amounts(
    base_scenario=scenario,
    amounts=[0, 50000, 100000, 150000, 200000],
    start_year=2026,
    end_year=2037,
    stop_at_rmd=True
)

fixed_optimizer_view

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(optimizer_view['bracket_top'], optimizer_view['total_tax'], marker='o')
plt.title('Lifetime Total Tax by Roth Conversion Bracket Target')
plt.xlabel('Bracket Top')
plt.ylabel('Total Tax')
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(optimizer_view['bracket_top'], optimizer_view['final_total_assets'], marker='o', label="Pre-Tax")
plt.plot(optimizer_view['bracket_top'], optimizer_view['final_after_tax_assets'], marker='o', label="After-Tax")

plt.legend()
plt.title('Pre-Tax vs After-Tax Assets by Roth Conversion Bracket Target')
plt.xlabel('Bracket Top')
plt.ylabel('Assets')
plt.grid(True)

plt.show()

plt.figure(figsize=(10, 5))
plt.plot(optimizer_view['bracket_top'], optimizer_view['max_rmd'], marker='o')
plt.title('Max RMD by Roth Conversion Bracket Target')
plt.xlabel('Bracket Top')
plt.ylabel('Max RMD')
plt.grid(True)
plt.show()

In [ ]:
df.loc[(df['year'] >= 2037) & (df['year'] <= 2042), [
    'year',
    'you_age',
    'gross_nonportfolio_income',
    'roth_conversion',
    'rmd_required',
    'withdrawal_traditional_cash',
    'gross_taxable_income',
    'taxable_income',
    'total_tax',
    'end_traditional',
    'end_roth'
]]

In [ ]:
df[[
    'year',
    'withdrawal_taxable',
    'taxable_principal_returned',
    'realized_capital_gain',
    'gross_capital_gains_income',
    'federal_capital_gains_tax',
    'total_tax'
]].head(15)

In [ ]:
df[[
    'year',
    'end_taxable',
    'end_taxable_cost_basis',
    'taxable_account_unrealized_gain'
]].head(15)

In [ ]:
df.loc[(df['year'] >= 2037) & (df['year'] <= 2042), [
    'year',
    'you_age',
    'rmd_required',
    'withdrawal_traditional_cash',
    'gross_ordinary_income',
    'gross_capital_gains_income',
    'total_tax',
    'end_traditional'
]]

In [ ]:
df[['year', 'rental_income', 'taxable_nonportfolio_income', 'gross_taxable_income', 'total_tax']].head(10)

In [ ]:
df[[
    'year',
    'ss_you',
    'ss_spouse',
    'combined_income',
    'taxable_social_security',
    'taxable_social_security_fraction',
    'gross_ordinary_income',
    'total_tax'
]].head(20)

In [ ]:
df.loc[(df['year'] >= 2028) & (df['year'] <= 2035), [
    'year',
    'ss_you',
    'ss_spouse',
    'combined_income',
    'taxable_social_security',
    'roth_conversion',
    'gross_ordinary_income',
    'total_tax'
]]

In [ ]:
# Clean annual tax view
tax_view = df[[
    'year',
    'you_age',
    'spouse_age',
    'salary_you',
    'salary_spouse',
    'rental_income',
    'pension_you',
    'pension_spouse',
    'ss_you',
    'ss_spouse',
    'gross_nonportfolio_income',
    'combined_income',
    'taxable_social_security',
    'taxable_social_security_fraction',
    'roth_conversion',
    'realized_capital_gain',
    'taxable_principal_returned',
    'rmd_required',
    'gross_ordinary_income',
    'gross_capital_gains_income',
    'gross_taxable_income',
    'deduction',
    'taxable_ordinary_income',
    'taxable_capital_gains',
    'taxable_income',
    'federal_ordinary_tax',
    'federal_capital_gains_tax',
    'federal_tax',
    'state_tax',
    'total_tax'
]].copy()

# Helper columns
tax_view['earned_income'] = tax_view['salary_you'] + tax_view['salary_spouse']
tax_view['pension_income'] = tax_view['pension_you'] + tax_view['pension_spouse']
tax_view['social_security_income'] = tax_view['ss_you'] + tax_view['ss_spouse']

# Reorder into a cleaner final layout
tax_view = tax_view[[
    'year',
    'you_age',
    'spouse_age',
    'earned_income',
    'rental_income',
    'pension_income',
    'social_security_income',
    'gross_nonportfolio_income',
    'roth_conversion',
    'realized_capital_gain',
    'rmd_required',
    'gross_ordinary_income',
    'gross_capital_gains_income',
    'gross_taxable_income',
    'deduction',
    'taxable_ordinary_income',
    'taxable_capital_gains',
    'taxable_income',
    'federal_ordinary_tax',
    'federal_capital_gains_tax',
    'federal_tax',
    'state_tax',
    'total_tax'
]]

tax_view.head(20)

In [ ]:
# Retirement planner dashboard view
planner_view = df[[
    'year',
    'you_age',
    'spouse_age',
    'gross_nonportfolio_income',
    'roth_conversion',
    'realized_capital_gain',
    'rmd_required',
    'total_spending',
    'total_tax',
    'withdrawal_cash',
    'withdrawal_taxable',
    'withdrawal_traditional_cash',
    'withdrawal_roth',
    'start_taxable_cost_basis',
    'end_taxable',
    'end_taxable_cost_basis',
    'taxable_account_unrealized_gain',
    'end_traditional',
    'end_roth',
    'end_cash',
    'end_total'
]].copy()

# Add cleaner grouped columns
planner_view['total_withdrawals_for_spending'] = (
    planner_view['withdrawal_cash']
    + planner_view['withdrawal_taxable']
    + planner_view['withdrawal_traditional_cash']
    + planner_view['withdrawal_roth']
)

planner_view['portfolio_total_ex_cash'] = (
    planner_view['end_taxable']
    + planner_view['end_traditional']
    + planner_view['end_roth']
)

# Reorder and rename for readability
planner_view = planner_view[[
    'year',
    'you_age',
    'spouse_age',
    'gross_nonportfolio_income',
    'roth_conversion',
    'rmd_required',
    'total_spending',
    'total_tax',
    'total_withdrawals_for_spending',
    'end_taxable',
    'taxable_account_unrealized_gain',
    'end_traditional',
    'end_roth',
    'end_cash',
    'end_total'
]].rename(columns={
    'you_age': 'your_age',
    'spouse_age': 'spouse_age',
    'gross_nonportfolio_income': 'income',
    'roth_conversion': 'roth_conversion',
    'rmd_required': 'rmd',
    'total_spending': 'spending',
    'total_tax': 'tax',
    'total_withdrawals_for_spending': 'withdrawals',
    'end_taxable': 'ending_taxable',
    'taxable_account_unrealized_gain': 'embedded_taxable_gain',
    'end_traditional': 'ending_traditional',
    'end_roth': 'ending_roth',
    'end_cash': 'ending_cash',
    'end_total': 'ending_total_assets'
})

planner_view

In [ ]:
# Yearly planner comparison dashboard

compare_yearly = pd.DataFrame({
    'year': df_none['year'],
    'end_total_no_conversion': df_none['end_total'],
    'end_total_fixed_100k': df_fixed_100k['end_total'],
    'end_total_fill_24': df_fill_24['end_total'],
    'traditional_no_conversion': df_none['end_traditional'],
    'traditional_fixed_100k': df_fixed_100k['end_traditional'],
    'traditional_fill_24': df_fill_24['end_traditional'],
    'roth_no_conversion': df_none['end_roth'],
    'roth_fixed_100k': df_fixed_100k['end_roth'],
    'roth_fill_24': df_fill_24['end_roth'],
    'tax_no_conversion': df_none['total_tax'],
    'tax_fixed_100k': df_fixed_100k['total_tax'],
    'tax_fill_24': df_fill_24['total_tax'],
    'rmd_no_conversion': df_none['rmd_required'],
    'rmd_fixed_100k': df_fixed_100k['rmd_required'],
    'rmd_fill_24': df_fill_24['rmd_required'],
    'roth_conv_no_conversion': df_none['roth_conversion'],
    'roth_conv_fixed_100k': df_fixed_100k['roth_conversion'],
    'roth_conv_fill_24': df_fill_24['roth_conversion'],
})

compare_yearly.head(20)

In [ ]:
def evaluate_spending_levels(base_scenario, spending_levels):
    results = []

    for spend in spending_levels:

        df_test = run_simulation(base_scenario, annual_spending=spend)

        results.append({
            'spending': spend,

            'final_total_assets': df_test['end_total'].iloc[-1],
            'final_after_tax_assets': compute_after_tax_total(df_test),

            'total_tax': df_test['total_tax'].sum(),
            'max_rmd': df_test['rmd_required'].max(),
        })

    return pd.DataFrame(results)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(spending_view['spending'], spending_view['final_after_tax_assets'], marker='o')

plt.title('After-Tax Assets vs Spending')
plt.xlabel('Annual Spending')
plt.ylabel('After-Tax Assets')
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(spending_view['spending'], spending_view['total_tax'], marker='o')

plt.title('Total Tax vs Spending')
plt.xlabel('Annual Spending')
plt.ylabel('Total Tax')
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(spending_view['spending'], spending_view['max_rmd'], marker='o')

plt.title('Max RMD vs Spending')
plt.xlabel('Annual Spending')
plt.ylabel('Max RMD')
plt.grid(True)

plt.show()

In [ ]:
# Scenario comparison charts

plt.figure(figsize=(10, 5))
plt.plot(df_none['year'], df_none['end_total'], label='No Conversion')
plt.plot(df_fixed_100k['year'], df_fixed_100k['end_total'], label='Fixed 100k/yr')
plt.plot(df_fill_24['year'], df_fill_24['end_total'], label='Fill 24% Bracket')
plt.title('Ending Total Assets by Scenario')
plt.xlabel('Year')
plt.ylabel('Assets')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_none['year'], df_none['end_traditional'], label='No Conversion')
plt.plot(df_fixed_100k['year'], df_fixed_100k['end_traditional'], label='Fixed 100k/yr')
plt.plot(df_fill_24['year'], df_fill_24['end_traditional'], label='Fill 24% Bracket')
plt.title('Traditional Balance by Scenario')
plt.xlabel('Year')
plt.ylabel('Traditional Balance')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_none['year'], df_none['end_roth'], label='No Conversion')
plt.plot(df_fixed_100k['year'], df_fixed_100k['end_roth'], label='Fixed 100k/yr')
plt.plot(df_fill_24['year'], df_fill_24['end_roth'], label='Fill 24% Bracket')
plt.title('Roth Balance by Scenario')
plt.xlabel('Year')
plt.ylabel('Roth Balance')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_none['year'], df_none['total_tax'], label='No Conversion')
plt.plot(df_fixed_100k['year'], df_fixed_100k['total_tax'], label='Fixed 100k/yr')
plt.plot(df_fill_24['year'], df_fill_24['total_tax'], label='Fill 24% Bracket')
plt.title('Annual Tax by Scenario')
plt.xlabel('Year')
plt.ylabel('Tax')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_none['year'], df_none['rmd_required'], label='No Conversion')
plt.plot(df_fixed_100k['year'], df_fixed_100k['rmd_required'], label='Fixed 100k/yr')
plt.plot(df_fill_24['year'], df_fill_24['rmd_required'], label='Fill 24% Bracket')
plt.title('RMD by Scenario')
plt.xlabel('Year')
plt.ylabel('RMD')
plt.legend()
plt.grid(True)
plt.show()

## 14. Summary Views

In [ ]:
summary_cols = [
    'year', 'you_age', 'spouse_age',
    'gross_nonportfolio_income', 'total_spending', 'total_tax',
    'rmd_required', 'roth_conversion',
    'end_taxable', 'end_traditional', 'end_roth', 'end_cash', 'end_total'
]

df[summary_cols].head(20)

## 15. Basic Charts

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df['year'], df['end_total'])
plt.title('Total Assets by Year')
plt.xlabel('Year')
plt.ylabel('End-of-Year Total Assets')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df['year'], df['end_taxable'], label='Taxable')
plt.plot(df['year'], df['end_traditional'], label='Traditional')
plt.plot(df['year'], df['end_roth'], label='Roth')
plt.plot(df['year'], df['end_cash'], label='Cash')
plt.title('Account Balances by Year')
plt.xlabel('Year')
plt.ylabel('Balance')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df['year'], df['gross_nonportfolio_income'], label='Income')
plt.plot(df['year'], df['total_spending'], label='Spending')
plt.plot(df['year'], df['total_tax'], label='Tax')
plt.title('Income vs Spending vs Tax')
plt.xlabel('Year')
plt.ylabel('Amount')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df['year'], df['rmd_required'], label='RMD Required')
plt.plot(df['year'], df['roth_conversion'], label='Roth Conversion')
plt.title('RMD and Roth Conversion by Year')
plt.xlabel('Year')
plt.ylabel('Amount')
plt.legend()
plt.grid(True)
plt.show()

## 16. Next Improvements

Good next upgrades after this starter version:
1. Replace sample inputs with your real inputs.
2. Improve Social Security taxation.
3. Improve state tax handling.
4. Add bracket-fill Roth conversion.
5. Separate your accounts and spouse accounts if needed.
6. Add scenario comparison.
7. Add IRMAA later.
8. Add Monte Carlo much later.